### Imports & Constants

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
import os

In [ ]:
import requests
from minsearch import Index

In [ ]:
## Model Constants
openai_client = OpenAI(
    base_url="http://127.0.0.1:1234/v1",
    # base_url="https://openrouter.ai/api/v1",
    # api_key=os.getenv("OPENROUTER_API_KEY"),
    api_key="1234",
)

In [ ]:
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

### Helper Functions

In [ ]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [ ]:
def llm_v1(prompt):
    response = openai_client.responses.create(
        model=os.getenv("OPENROUTER_MODEL"),
        input=prompt
    )
    return response.output_text

def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=os.getenv("OPENROUTER_MODEL", model),
        input=message_history
    )

    return response.output_text

def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )
def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

### Experimentation

In [ ]:
llm("Hey, what's up?")

In [ ]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

In [ ]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [ ]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [ ]:
documents[:2]

In [ ]:
search_results = search(question)

In [ ]:
search_results

In [ ]:
answer = rag("I just discovered the course. Can I join now?")
answer

In [ ]:
rag("How do I get a certificate?")

### RAG Flow

In [42]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from utilities import calculate_llm_price

from openai import OpenAI
from sqlitesearch import TextSearchIndex

In [3]:
openai_client = OpenAI(
    base_url=os.getenv("OPENROUTER_URL"),
    # model=os.getenv(""),
    api_key=os.getenv("OPENROUTER_API_KEY")
)

In [ ]:
index = build_index()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    model=os.getenv("OPENROUTER_MODEL")
)

In [2]:
sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [17]:
answer = assistant.rag("I just discovered the course. Can I join now?")
answer

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [18]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
"""

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=custom_instructions,
)

In [19]:
assistant.rag("How do I get a certificate?")
# assistant.rag("Can I still join the course after it started?")

'To get a certificate, you must finish the course with a "live" cohort, as certificates are not awarded for the self-paced mode. You also need to pass the Capstone project and peer-review 3 capstone(s) after submitting your project.'

In [20]:
assistant.rag("Can I still join the course after it started?")

'Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start the course whenever you want as the videos and GitHub materials are available.'

In [4]:
assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

In [5]:
answer = assistant.rag("I just discovered the course. Can I join now?")
answer

'Yes, you can join. However, if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.'

In [6]:
search_tool = {
    "type": "function",
    "name": "assistant.search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [31]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
    reasoning={"effort": "none"}
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"joining the course"}', call_id='call_1492548176320001', name='assistant.search', type='function_call', id='fc_0do1zuz179ildbp5blktnn', namespace=None, status='completed')]

In [32]:
dict(response)

{'id': 'resp_fef3d78f056cd3183949ae3d126566fde77bea9ee0fdd9bc',
 'created_at': 1782031769.0,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'google/gemma-4-12b-qat',
 'object': 'response',
 'output': [ResponseFunctionToolCall(arguments='{"query":"joining the course"}', call_id='call_1492548176320001', name='assistant.search', type='function_call', id='fc_0do1zuz179ildbp5blktnn', namespace=None, status='completed')],
 'parallel_tool_calls': True,
 'temperature': 1.0,
 'tool_choice': 'auto',
 'tools': [FunctionTool(name='assistant.search', parameters={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query text to look up in the course FAQ.'}}, 'required': ['query'], 'additionalProperties': False}, strict=False, type='function', defer_loading=None, description='Search the FAQ database for entries matching the given query.')],
 'top_p': 0.95,
 'background': False,
 'completed_at': 1782031770.0,
 'conversatio

In [33]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = assistant.search(**args)
result_json = json.dumps(results, indent=2)

In [34]:
result_json

'[\n  {\n    "id": "bd31146b0e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "When will the course be offered next?",\n    "answer": "Summer 2027."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n    "answer": "No, you can only get a certificate if you finish the course with a \\"live\\" cohort.\\n\\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\\n\\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."\n  },\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "a

In [35]:
messages.extend(response.output)
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"joining the course"}', call_id='call_1492548176320001', name='assistant.search', type='function_call', id='fc_0do1zuz179ildbp5blktnn', namespace=None, status='completed')]

In [36]:
messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"joining the course"}', call_id='call_1492548176320001', name='assistant.search', type='function_call', id='fc_0do1zuz179ildbp5blktnn', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_1492548176320001',
  'output': '[\n  {\n    "id": "bd31146b0e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "When will the course be offered next?",\n    "answer": "Summer 2027."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n    "answer": "No, you can only get a certificate if you finish the course with a \\"live\\" cohort.\\n\\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-rev

In [37]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes, you can still join the course!\n\nHowever, please keep in mind that if you want to receive a certificate, you must submit your project while submissions are still being accepted.'

In [38]:
dict(response)

{'id': 'resp_bc9f96eb49c2810eeaf1e87276106f847b9ca396deea3b0c',
 'created_at': 1782031931.0,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'google/gemma-4-12b-qat',
 'object': 'response',
 'output': [ResponseReasoningItem(id='rs_v6g2tgbdqgrk4dhwpqxapr', summary=[], type='reasoning', content=[Content(text='\n', type='reasoning_text')], encrypted_content=None, status='completed'),
  ResponseOutputMessage(id='msg_5s2u1ug9pe7ubyxgatljv', content=[ResponseOutputText(annotations=[], text='Yes, you can still join the course!\n\nHowever, please keep in mind that if you want to receive a certificate, you must submit your project while submissions are still being accepted.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)],
 'parallel_tool_calls': True,
 'temperature': 1.0,
 'tool_choice': 'auto',
 'tools': [FunctionTool(name='assistant.search', parameters={'type': 'object', 'properties': {'quer

In [43]:
usage = response.usage
usage

ResponseUsage(input_tokens=775, input_tokens_details=InputTokensDetails(cached_tokens=86), output_tokens=38, output_tokens_details=OutputTokensDetails(reasoning_tokens=1), total_tokens=813)

In [44]:
price_report = calculate_llm_price(usage.input_tokens, usage.output_tokens, model_name=response.model)
price_report

{'input_cost': 0.00011624999999999999,
 'output_cost': 2.2800000000000002e-05,
 'total_cost': 0.00013905,
 'model_name': 'google/gemma-4-12b-qat'}

In [45]:
print("Total cost: $", round(price_report["total_cost"], 8))

Total cost: $ 0.00013905
